<a href="https://colab.research.google.com/github/Qureshiii/PyTorch-Learning-Journey/blob/main/08_Automated_Hyperparameter_Optimization_Optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [ ]:
torch.manual_seed(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {device}')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/fashion-mnist_train.csv')
df.head()

In [ ]:
# train test split
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


X_train = X_train/255.0
X_test = X_test/255.0
X_train

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):

    return len(self.features)

  def __getitem__(self, index):

    return self.features[index], self.labels[index]

In [ ]:
train_dataset = CustomDataset(X_train, y_train)

# just to verify

len(train_dataset)

test_dataset = CustomDataset(X_test, y_test)

# just to verify

len(test_dataset)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [ ]:
class MyNN(nn.Module):

  def __init__(self, input_dims, output_dims, num_hidden_layers, neurons_per_layer, dropout_rate):

    super().__init__()

    layers = []

    for i in range(num_hidden_layers):
      layers.append(nn.Linear(input_dims, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dims = neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer, output_dims))


    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.model(x)

### Creating Objective function

In [ ]:
def objective(trial):

    # 1. Hyper-parameter search space definition
    num_hidden_layers = trial.suggest_int('num_hidden_layers', 1, 5)
    neurons_per_layer = trial.suggest_int('neurons_per_layer', 8, 128, step=8)
    epochs = trial.suggest_int('epochs', 10, 50, step=10)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float('Dropout_rate', 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical('Batch_size', [16, 32, 64, 128])
    optimizer_name = trial.suggest_categorical('optimizer_name', ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    # 2. Dynamic Dataloaders according to Optuna Batch Size
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    # 3. Model initialization & explicit GPU migration
    input_dims = 784
    output_dims = 10
    model = MyNN(input_dims, output_dims, num_hidden_layers, neurons_per_layer, dropout_rate)
    model.to(device) # Transfers full weights to GPU memory

    criterion = nn.CrossEntropyLoss()

    # 4. Dynamic Optimizer assignment fixed
    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # 5. Training loop (GPU Accelerated)
    for epoch in range(epochs):
        model.train() # Set model to training mode (BatchNorm/Dropout dynamic activate)
        for batch_features, batch_labels in train_loader:
            # Explicitly migrate input mini-batches to GPU device
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

            # Execution Pipeline
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # 6. Evaluation loop (GPU Accelerated)
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for batch_features, batch_labels in test_loader:
            # Transfer evaluation batches to GPU
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)

            total += batch_labels.shape[0]
            correct += (predicted == batch_labels).sum().item()

    accuracy = (correct / total) * 100
    return accuracy


### Creating Study

In [ ]:
import optuna

study = optuna.create_study(direction='maximize')

In [ ]:
study.optimize(objective, n_trials=10)

In [ ]:
study.best_value

In [ ]:
study.best_params